# Process Simulation & Optimization (DWSIM)
Analysis of trial runs from a simple **DWSIM** flowsheet. In DWSIM you vary key operating parameters, record the results, and this notebook shows each parameter's effect on **efficiency** and **output** and identifies the best-performing settings.

**Data:** save your trial log as `data/dwsim_trials.csv` with columns
`trial, feed_temp_C, pressure_kPa, reflux_ratio, efficiency_pct, output_kg_h`.
Rename parameters in `PARAMS` to match your flowsheet. If the file is missing, **synthetic demo data** is generated so the notebook runs.

> DWSIM itself is a GUI simulator; this notebook analyses the numbers you export from it.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PARAMS = ["feed_temp_C", "pressure_kPa", "reflux_ratio"]
RESPONSES = ["efficiency_pct", "output_kg_h"]

## 1. Load trial log

In [ ]:
PATH = "data/dwsim_trials.csv"
if os.path.exists(PATH):
    trials = pd.read_csv(PATH)
    print("Loaded real trials:", trials.shape)
else:
    rng = np.random.default_rng(7)
    base = {"feed_temp_C": 60, "pressure_kPa": 150, "reflux_ratio": 2.0}
    levels = {"feed_temp_C": [40, 50, 60, 70, 80],
              "pressure_kPa": [100, 125, 150, 175, 200],
              "reflux_ratio": [1.0, 1.5, 2.0, 2.5, 3.0]}
    recs = [dict(base)]
    for p in PARAMS:                       # one-factor-at-a-time trials
        for v in levels[p]:
            if v != base[p]:
                r = dict(base); r[p] = v; recs.append(r)
    trials = pd.DataFrame(recs)
    t, p, r = trials.feed_temp_C, trials.pressure_kPa, trials.reflux_ratio
    trials["efficiency_pct"] = (78 - 0.004*(t-65)**2 - 0.0006*(p-155)**2 - 3.0*(r-2.2)**2
                                + rng.normal(0, 0.4, len(trials))).round(2)
    trials["output_kg_h"] = (520 + 1.8*(t-60) - 0.9*(p-150) - 25*(r-2.0)
                             + rng.normal(0, 4, len(trials))).round(1)
    trials.insert(0, "trial", range(1, len(trials) + 1))
    print("Using SYNTHETIC demo data:", trials.shape)
trials

## 2. Effect of each parameter

In [ ]:
fig, axes = plt.subplots(len(RESPONSES), len(PARAMS), figsize=(13, 7))
for i, resp in enumerate(RESPONSES):
    for j, par in enumerate(PARAMS):
        ax = axes[i, j]
        ax.scatter(trials[par], trials[resp], alpha=0.8)
        g = trials.groupby(par)[resp].mean()
        ax.plot(g.index, g.values, color="red", lw=1)
        ax.set_xlabel(par); ax.set_ylabel(resp)
plt.tight_layout(); plt.show()

In [ ]:
# change in response per unit change in each parameter (linear slope) and correlation
rows = []
for par in PARAMS:
    for resp in RESPONSES:
        slope = np.polyfit(trials[par], trials[resp], 1)[0]
        rows.append({"parameter": par, "response": resp, "slope_per_unit": slope,
                     "correlation": trials[par].corr(trials[resp])})
effects = pd.DataFrame(rows).round(3)
effects

## 3. Best-performing settings

In [ ]:
best_eff = trials.loc[trials["efficiency_pct"].idxmax()]
best_out = trials.loc[trials["output_kg_h"].idxmax()]
print("Best EFFICIENCY trial:\n", best_eff.to_string(), "\n")
print("Best OUTPUT trial:\n", best_out.to_string())

# simple trade-off score: equal-weight, min-max scaled
s = trials.copy()
for r in RESPONSES:
    s[r + "_scaled"] = (s[r] - s[r].min()) / (s[r].max() - s[r].min())
s["score"] = s[[r + "_scaled" for r in RESPONSES]].mean(axis=1)
print("\nBest balanced trial (equal weights):")
s.loc[s["score"].idxmax(), ["trial"] + PARAMS + RESPONSES + ["score"]]

## 4. Efficiency vs output trade-off

In [ ]:
plt.figure(figsize=(7, 5))
sc = plt.scatter(trials["output_kg_h"], trials["efficiency_pct"], c=trials["reflux_ratio"], cmap="viridis")
plt.colorbar(sc, label="reflux_ratio")
for _, r in trials.iterrows():
    plt.annotate(int(r["trial"]), (r["output_kg_h"], r["efficiency_pct"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
plt.xlabel("Output (kg/h)"); plt.ylabel("Efficiency (%)"); plt.title("Trial trade-off")
plt.tight_layout(); plt.show()

## 5. Summary

In [ ]:
print(f"- {len(trials)} trials analysed across {len(PARAMS)} parameters.")
print(f"- Highest efficiency: {best_eff['efficiency_pct']} % at", {p: float(best_eff[p]) for p in PARAMS})
print(f"- Highest output: {best_out['output_kg_h']} kg/h at", {p: float(best_out[p]) for p in PARAMS})
print("- Next step: re-run the best settings in DWSIM to confirm, then refine around them with a finer grid.")